# Dataset and ExperimentData

[All Quick Tutorials](README.md)

A role tells HypEx how to use a column. `TreatmentRole` identifies group assignments and `TargetRole` identifies an outcome to compare. The same column can be an outcome in one analysis and a historical covariate in another.

Prepare data for an A/B test. `Treatment` contains group labels, `Target` is measured after treatment, and `Target_lag_1` is measured in the previous period.

## Contents

- [How roles define experiment inputs](#dataset-theory)
- [Inspect the input data](#dataset-input)
- [Assign roles and choose a backend](#dataset-roles)
- [Compress a Spark DataFrame when loading it into pandas](#dataset-compression)
- [Find a metric in ExperimentData](#dataset-search)

<a id="dataset-theory"></a>

## How roles define experiment inputs

Let $\mathcal C$ be the set of columns and $r(c)$ the role assigned to column $c$. For a required role $R$, an experiment can select the set

$$
\mathcal C_R=\{c\in\mathcal C\mid r(c)=R\}.
$$

For example, `TargetRole` identifies the metrics to compare. You can assign different roles to the same table for different analyses.

<a id="dataset-input"></a>

## Inspect the input data

Pass a pandas or Spark DataFrame, a CSV path, or `None` for an empty dataset as `data`. This example starts with a pandas DataFrame. Its first rows show the group assignments and metric values for two periods.

In [1]:
from hypex.utils import BackendsEnum
from hypex.dataset import (
    Dataset,
    ExperimentData,
    InfoRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=1200, dependent_division=False, random_state=42
)
frame[["Treatment", "Target", "Target_lag_1"]].head(3)

,Treatment,Target,Target_lag_1
0,0,1093.358960,389.946418
1,1,725.459061,247.270884
2,0,853.178392,1120.783778


`Treatment` contains the group label. `Target` is the outcome, and `Target_lag_1` is the metric from the previous period.

<a id="dataset-roles"></a>

## Assign roles and choose a backend

`roles` can map column names to roles or map a role to one or more column names. Set `default_role=InfoRole()` for the remaining columns. With the default `default_role=None`, HypEx assigns them `DefaultRole`.

A role's `data_type` sets its expected type, as in `TreatmentRole(int)`. With `None`, HypEx infers the type from the data. For `PreTargetRole` and `FeatureRole`, `parent` names the original variable and `lag` identifies the historical period. `cofounders` on `TargetRole` and `PreTargetRole` names the covariates for CUPAC.

When `backend=None`, the choice depends on the input and the session. You can pass a `SparkSession` through `session` for Spark.

The example spells out `backend=BackendsEnum.pandas`. HypEx infers this backend from a pandas DataFrame. `default_role=InfoRole()` applies only to columns without an explicit role.

In [2]:
ab_data = Dataset(
    roles={
        "Treatment": TreatmentRole(int),
        "Target": TargetRole(float),
        "Target_lag_1": PreTargetRole(parent="Target", lag=1),
    },
    data=frame,
    backend=BackendsEnum.pandas,
    default_role=InfoRole(),
)
ab_data.roles["Target_lag_1"], ab_data.roles["X"]

(PreTarget(data_type=<class 'float'>, parent='Target', lag=1),
 Info(<class 'float'>))

The output shows `parent='Target'` and `lag=1` for `Target_lag_1` and the `Info` role for the auxiliary column `X`.

<a id="dataset-compression"></a>

## Compress a Spark DataFrame when loading it into pandas

Pass a Spark SQL DataFrame directly to `Dataset` with `backend=BackendsEnum.pandas`. With the default `data_compression="auto"`, HypEx downcasts `double` and `decimal` columns and encodes eligible strings as integers on Spark before collecting the table to the driver.

The final table uses less memory. Budget driver memory for the full result and temporary data used during collection. The [compression guide](../how_to_guide/spark-to-pandas-compression.ipynb) compares each mode and measures memory on a worked example.

<a id="dataset-search"></a>

## Find a metric in ExperimentData

`ExperimentData(data)` accepts a `Dataset` or `SmallDataset`. It keeps the input columns and intermediate results. Standard experiments accept a `Dataset` directly. Here, use `ExperimentData` to find the outcome among the input columns.

In [3]:
experiment_data = ExperimentData(ab_data)
experiment_data.field_search(TargetRole(), space="ds")

['Target']

The search returned `['Target']`. The historical column has `PreTargetRole` and can be used as a covariate for adjustment.

Choose an experiment from [Quick Tutorials](README.md).

See how to [create a Spark Dataset with the same roles](SparkBackend.ipynb#spark-dataset).